Small experiments testing resampling methods on image data.

In [ ]:
import copy

import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
dataset ="mnist"
architecture = "resnet_small"
budget = 120

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset
from src.utils.transforms.apply import grid_resample
dataset_info = get_dataset_info(dataset)

#dataset_info.transform_seq_name = "mnist_reflection"
dataset_info.resample_method = grid_resample
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data)


In [ ]:
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
from src.utils.transforms.apply import (
    grid_resample_bilinear,
    grid_resample_nearest,
    grid_resample_bicubic,
    grid_resample_deblur,
)

resample_methods = {
    "bilinear": grid_resample_bilinear,
    "nearest": grid_resample_nearest,
    "bicubic": grid_resample_bicubic,
    "deblur": grid_resample_deblur,
}

transformed_loaders = {}

for name, method in resample_methods.items():
    info = get_dataset_info(dataset)
    info.sample_method = method
    info.resample_method = method
    dset_dict = get_dataset(info, path=experiment_files_path_data)
    transformed_loaders[name] = dset_dict




# Example: Access the transformed loader for 'nearest'
train_loader_transformed_bilinear = transformed_loaders["bilinear"]["train_loader_transformed"]
train_loader_transformed_nearest = transformed_loaders["nearest"]["train_loader_transformed"]
train_loader_transformed_bicubic = transformed_loaders["bicubic"]["train_loader_transformed"]
train_loader_transformed_deblur = transformed_loaders["deblur"] ["train_loader_transformed"]

val_loader_transformed_bilinear  = transformed_loaders["bilinear"]["val_loader_transformed"]
val_loader_transformed_nearest = transformed_loaders["nearest"]["val_loader_transformed"]
val_loader_transformed_bicubic = transformed_loaders["bicubic"]["val_loader_transformed"]
val_loader_transformed_deblur = transformed_loaders["deblur"]["val_loader_transformed"]

test_loader_transformed_bilinear = transformed_loaders["bilinear"]["test_loader_transformed"]
test_loader_transformed_nearest = transformed_loaders["nearest"]["test_loader_transformed"]
test_loader_transformed_bicubic = transformed_loaders["bicubic"]["test_loader_transformed"]
test_loader_transformed_deblur = transformed_loaders["deblur"]["test_loader_transformed"]


In [ ]:
test_loader_transformed=None

In [ ]:
#print examples from transformed loaders
import torchvision
from src.utils.eval.vis import vis_dataset, plt_setup_latex

vis_dataset(train_loader,val_loader,test_loader_transformed_bilinear)
vis_dataset(train_loader_transformed_nearest,val_loader_transformed_nearest,test_loader_transformed_nearest)



In [ ]:
vis_dataset(train_loader_transformed_bicubic,val_loader_transformed_bicubic,test_loader_transformed_bicubic)
vis_dataset(train_loader_transformed_deblur,val_loader_transformed_deblur,test_loader_transformed_deblur)

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]

In [ ]:
from model.train import train_and_get_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")

model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": 100,
        "precision": "16-mixed",
},load_if_exists=True)


res = evaluate_base_model(model, test_loader_transformed_bilinear, device)
print(res)

In [ ]:
from data.transformation import get_transformation_sequence_images
if dataset_info.transform_seq_name is not None and dataset_info.datatype == "image":
    transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()

In [ ]:
model

In [ ]:
from torchinfo import summary
summary(model, (1,1, 28, 28))

In [ ]:
from src.utils.eval.ood_performance import ITSWRAPPER, load_or_run_evaluate_confidence_and_search
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

def plot_transformed_batch(data_loader, problem, optimizer, model, k=8, device=None):
    """
    Fetches the first batch, applies transformation, and plots the first k samples
    side by side: original (before) and transformed (after), with green/red border for correct/incorrect classification.
    """
    def add_border(ax, color, lw=4):
        rect = patches.Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            fill=False,
            edgecolor=color,
            linewidth=lw,
            zorder=3,
            clip_on=False
        )
        ax.add_patch(rect)

    with torch.no_grad():
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        data_iter = iter(data_loader)
        data, labels = next(data_iter)
        data = data[:k].to(device)
        labels = labels[:k].to(device)

        if not isinstance(optimizer, ITSWRAPPER):
            res = optimizer.optimize(problem, data, y=None)
            x_trans = problem.transform(data, res[0])
        else:
            x_trans, _ = optimizer.optimize(problem, data)

        x_trans_cpu = x_trans.detach().cpu()
        data_cpu = data.detach().cpu()
        labels_cpu = labels.detach().cpu()

        model.eval()
        preds_orig = model(data).argmax(dim=1).cpu()
        preds_trans = model(x_trans).argmax(dim=1).cpu()

        plt.figure(figsize=(k * 2, 4))
        for i in range(min(k, x_trans_cpu.shape[0])):
            # Before
            ax1 = plt.subplot(2, k, i + 1)
            img = data_cpu[i]
            if img.dim() == 3 and img.shape[0] in [1, 3]:
                img = img.permute(1, 2, 0)
            ax1.imshow(img.squeeze(), cmap='gray' if img.shape[-1] == 1 else None)
            ax1.set_xticks([]); ax1.set_yticks([])
            add_border(ax1, 'green' if preds_orig[i] == labels_cpu[i] else 'red')
            if i == 0:
                ax1.set_title('Before')

            # After
            ax2 = plt.subplot(2, k, k + i + 1)
            img_t = x_trans_cpu[i]
            if img_t.dim() == 3 and img_t.shape[0] in [1, 3]:
                img_t = img_t.permute(1, 2, 0)
            ax2.imshow(img_t.squeeze(), cmap='gray' if img_t.shape[-1] == 1 else None)
            ax2.set_xticks([]); ax2.set_yticks([])
            add_border(ax2, 'green' if preds_trans[i] == labels_cpu[i] else 'red')
            if i == 0:
                ax2.set_title('After')

        plt.tight_layout()
        plt.show()

In [ ]:
from search import random_search

random_search_large = random_search.RSLR(initial_samples=240, local_runs=1, local_max_steps=0, project_param=True)


In [ ]:
#plot_transformed_batch(test_loader_transformed, problem_nn_pytorch_pretrained, random_search_large,model, k=32, device=device)


In [ ]:
random_search_60 = random_search.RSLR(initial_samples=60, local_runs=1, local_max_steps=0, project_param=True)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from torch.utils.data import SequentialSampler
from embedding_cache import LayerEmbeddingCache

transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train2"

cache_train = LayerEmbeddingCache(model, train_loader_no_shuffle,
                                  cache_dir=os.path.join(embedding_cache_path, cache_name_train))

embeddings_t, final_t, classes_t = cache_train("fc", capture_modes='input', flatten=True)
dual_output_model = cache_train.make_wrapper("fc", capture_modes='input', concat=False, flatten=True)

inter, final = dual_output_model(torch.randn(2, 1, 28, 28).to(device))


from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence



nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine")
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple")


In [ ]:
from src.utils.eval.ood_performance import evaluate_confidence_and_search,load_or_run_evaluate_confidence_and_search

test_loaders_dict = {
    "original": test_loader,
    "bilinear": test_loader_transformed_bilinear,
    "nearest": test_loader_transformed_nearest,
    "bicubic": test_loader_transformed_bicubic,
    "deblur": test_loader_transformed_deblur,
}

for name, loader in test_loaders_dict.items():
    print(f"Evaluating on {name} data:")
    save_path = os.path.join(current_path, "experiment_files", "results", "experiment_resampling_method",name)
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=random_search_60, problem=problem_nn_pytorch_pretrained,
        test_loader=loader,
        max_batch_override=1024, repeats=10,
        save_path=save_path
    )
    print(res)
    print("\n")


In [ ]:
import src.utils.transforms.apply
import importlib
importlib.reload(src.utils.transforms.apply)

from src.utils.transforms.apply import grid_resample,grid_resample_bicubic,grid_resample_nearest,grid_resample_blur_simple,grid_resample_border,grid_resample_blur,grid_resample_reflection,grid_resample_deblur,AdjustedGridResample

background_color = (-0.25,)

list_of_resample_methods = [AdjustedGridResample(background_color=background_color),grid_resample_bilinear,grid_resample_bicubic,grid_resample_nearest,grid_resample_blur_simple,grid_resample_deblur]
names_of_resample_methods = ["adjusted","bilinear","bicubic","nearest","blur_simple","deblur"]

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
#now iterate over all resample methods and evaluate. We only change this during the problem
for resample_method, name in zip(list_of_resample_methods,names_of_resample_methods):
    print(f"Evaluating on {name} data:")
    problem = TransformationProblem(conf_mod_nn_pytorch_pretrained, get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=resample_method
    ).cuda(),
                                                      consolidate_method="consolidate_simple")
    problem.max_batch_size=2048
    path = os.path.join(current_path, "experiment_files", "results", "experiment_resampling_method_5",name)
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=random_search_60, problem=problem,
        test_loader=test_loader,
        max_batch_override=2048, repeats=10,
        save_path=path
    )
    print(res)
    print("\n")

In [ ]:
#now iterate over all resample methods and evaluate. We only change this during the problem
for resample_method, name in zip(list_of_resample_methods,names_of_resample_methods):
    print(f"Evaluating on {name} data:")
    problem = TransformationProblem(conf_mod_nn_pytorch_pretrained, get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=resample_method
    ).cuda(),
                                                      consolidate_method="consolidate_simple")
    problem.max_batch_size=2048
    path = os.path.join(current_path, "experiment_files", "results", "experiment_resampling_method_6",name)
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=random_search_60, problem=problem,
        test_loader=test_loader_transformed_nearest,
        max_batch_override=2048, repeats=10,
        save_path=path
    )
    print(res)
    print("\n")

In [ ]:
#now iterate over all resample methods and evaluate. We only change this during the problem
for resample_method, name in zip(list_of_resample_methods,names_of_resample_methods):
    print(f"Evaluating on {name} data:")
    problem = TransformationProblem(conf_mod_nn_pytorch_pretrained, get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=resample_method
    ).cuda(),
                                                      consolidate_method="consolidate_simple")
    problem.max_batch_size=2048
    path = os.path.join(current_path, "experiment_files", "results", "experiment_resampling_method_2",name)
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=random_search_60, problem=problem,
        test_loader=test_loader_transformed_deblur,
        max_batch_override=2048, repeats=10,
        save_path=path
    )
    print(res)
    print("\n")

In [ ]:
#now iterate over all resample methods and evaluate. We only change this during the problem
for resample_method, name in zip(list_of_resample_methods,names_of_resample_methods):
    print(f"Evaluating on {name} data:")
    problem = TransformationProblem(conf_mod_nn_pytorch_pretrained, get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=resample_method
    ).cuda(),
                                                      consolidate_method="consolidate_simple")
    problem.max_batch_size=2048
    path = os.path.join(current_path, "experiment_files", "results", "experiment_resampling_method_3",name)
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=random_search_60, problem=problem,
        test_loader=test_loader_transformed_bicubic,
        max_batch_override=2048, repeats=10,
        save_path=path
    )
    print(res)
    print("\n")

In [ ]:
#now iterate over all resample methods and evaluate. We only change this during the problem
for resample_method, name in zip(list_of_resample_methods,names_of_resample_methods):
    print(f"Evaluating on {name} data:")
    problem = TransformationProblem(conf_mod_nn_pytorch_pretrained, get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=resample_method
    ).cuda(),
                                                      consolidate_method="consolidate_simple")
    problem.max_batch_size=2048
    path = os.path.join(current_path, "experiment_files", "results", "experiment_resampling_method_4",name)
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=random_search_60, problem=problem,
        test_loader=test_loader_transformed_bilinear,
        max_batch_override=2048, repeats=10,
        save_path=path
    )
    print(res)
    print("\n")

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------
# Load accuracy_mean and SE from all experiments
# -------------------------------

experiment_dirs = [
    ("Original", "experiment_files/results/experiment_resampling_method_5"),
    ("Deblur", "experiment_files/results/experiment_resampling_method_2"),
    ("Bicubic", "experiment_files/results/experiment_resampling_method_3"),
    ("Bilinear", "experiment_files/results/experiment_resampling_method_4"),
    ("Nearest", "experiment_files/results/experiment_resampling_method_6"),
]

tables = []
filter_list = ["adjusted","blur_simple"]



for exp_label, rel_dir in experiment_dirs:
    full_dir = os.path.join(current_path, rel_dir)
    if not os.path.exists(full_dir):
        print(f"Warning: {full_dir} does not exist, skipping.")
        continue

    exp_data = []
    for fname in os.listdir(full_dir):
        file_path = os.path.join(full_dir, fname)
        if not os.path.isfile(file_path):
            continue
        method_name = os.path.splitext(fname)[0]
        if method_name in filter_list:
            continue
        try:
            with open(file_path, "r") as f:
                res = json.load(f)
            # Extract mean and SE
            if "accuracy_mean" in res and "accuracy_se" in res:
                exp_data.append({
                    "Method": method_name,
                    "Accuracy": res["accuracy_mean"],
                    "SE": res["accuracy_se"]
                })
        except Exception as e:
            print(f"Failed to load {file_path}: {e}")

    df_exp = pd.DataFrame(exp_data)

    # -------------------------------
    # Filter and rename methods per experiment
    # -------------------------------
    #if exp_label == "Dataset Sampling":
    #    # Remove 'original', rename 'transformed'  'bilinear'
    #    df_exp = df_exp[df_exp["Method"] != "original"]
    #    df_exp["Method"] = df_exp["Method"].replace({"transformed": "bilinear"})
    #else:
    #    # For search experiments: remove blur, adjusted, border, reflection, rename
    #    df_exp = df_exp[~df_exp["Method"].isin(["blur", "adjusted", "border", "reflection"])]
    #    df_exp["Method"] = df_exp["Method"].replace({"blur_simple": "blur", "original": "bilinear"})

    #rename dataset sampling to untransformed data
    #if exp_label == "Dataset Sampling":
    #    exp_label = "Untransformed Data"

    df_exp = df_exp.sort_values("Accuracy", ascending=False)
    tables.append((exp_label, df_exp))
    print(f"\n=== {exp_label} ===")
    print(df_exp)

# -------------------------------
# Plot three experiments side by side with error bars
# -------------------------------




In [ ]:
W = plt_setup_latex()

In [ ]:
figure_path = os.path.join(current_path,"experiment_files","export","fig","resampling_method")

In [ ]:
figure_path

In [ ]:
aspect = 7/3

In [ ]:
#remove original from talbes
filtered_tables = []
for label, df in tables:
    #capitalize method
    df["Method"] = df["Method"].str.title()

    if label != "Original":
        filtered_tables.append((label, df))




tables = filtered_tables




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Settings
plt.style.use("default")
error_capsize = 3

# Prepare figure (no constrained_layout, well control layout manually)
fig, axes = plt.subplots(
    1, len(tables),
    figsize=(W, W/2.9),
    sharey=True
)

# Handle single subplot case
if len(tables) == 1:
    axes = [axes]

# --- Compute global y-limits so all subplots use same scale ---
global_min = float('inf')
global_max = -float('inf')
for _, df_exp in tables:
    local_min = (df_exp["Accuracy"] - 1.96 * df_exp["SE"]).min()
    local_max = (df_exp["Accuracy"] + 1.96 * df_exp["SE"]).max()
    global_min = min(global_min, local_min)
    global_max = max(global_max, local_max)

# Add a small uniform margin
margin = (global_max - global_min) * 0.1
ymin, ymax = global_min - margin, global_max + margin

# --- Plot each experiment ---
for ax, (exp_label, df_exp) in zip(axes, tables):
    x_pos = np.arange(len(df_exp))

    bars = ax.bar(
        x_pos,
        df_exp["Accuracy"],
        yerr=df_exp["SE"],
        capsize=error_capsize,
        color=sns.color_palette("viridis", len(df_exp)),
        edgecolor="none",
        width=0.7
    )

    # Axes formatting
    ax.set_xticks(x_pos)
    ax.set_xticklabels(df_exp["Method"], rotation=35, ha="right", fontsize=8)
    ax.set_title(exp_label, fontsize=11)
    ax.tick_params(axis='y', labelsize=9)
    ax.set_ylim(ymin, ymax)
    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    sns.despine(ax=ax)

# Y-label on leftmost plot only
axes[0].set_ylabel("Accuracy", fontsize=10)

# Tight layout  works properly without constrained_layout
plt.tight_layout(pad=0.5, w_pad=0.2)

# Save
os.makedirs(figure_path, exist_ok=True)
fig.savefig(os.path.join(figure_path, "resampling_method.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(figure_path, "resampling_method.pgf"), bbox_inches="tight")

plt.show()


In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

# --- Define output directory ---
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "resampling_method")
os.makedirs(figure_path, exist_ok=True)

# --- Get one example from each loader ---
index = 0
x_untransformed = next(iter(test_loader))[0][index]
x_bil  = next(iter(test_loader_transformed_bilinear))[0][index]
x_near = next(iter(test_loader_transformed_nearest))[0][index]
x_bicub = next(iter(test_loader_transformed_bicubic))[0][index]
x_deblur = next(iter(test_loader_transformed_deblur))[0][index]

# --- Convert tensors to numpy arrays for plotting ---
def to_numpy(img):
    if torch.is_tensor(img):
        img = img.detach().cpu()
        if img.ndim == 3 and img.shape[0] in [1, 3]:
            img = img.permute(1, 2, 0)
        img = img.numpy()
    return img.squeeze()

# --- Upscale with nearest-neighbor only ---
def upscale_tensor(img, scale=5):
    if torch.is_tensor(img):
        img = img.unsqueeze(0)  # Add batch dim
        upscaled = F.interpolate(img, scale_factor=scale, mode="nearest")
        return upscaled.squeeze(0)
    return img

# --- Dictionary of all images ---
images = {
    "original": x_untransformed,
    "nearest": x_near,
    "bilinear": x_bil,
    "bicubic": x_bicub,
    "deblurred": x_deblur
}

# --- Save each image as upscaled PNG ---
for name, img in images.items():
    img_up = upscale_tensor(img, scale=5)

    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(to_numpy(img_up), cmap="gray")
    ax.axis("off")

    out_path = os.path.join(figure_path, f"{name}.png")
    plt.savefig(out_path, bbox_inches="tight", pad_inches=0, dpi=300)
    plt.show(fig)

print(f" Saved {len(images)} PNGs (nearest-upscaled) to: {figure_path}")
